# 第6章：高级数据处理

## 本章学习目标

- 掌握 PIT 数据的使用
- 理解数据重采样机制
- 实现滚动数据处理
- 了解性能优化技巧

---

## 6.1 PIT 数据概述

PIT (Point-in-Time) 数据是指在特定时间点可获得的真实数据，避免了前视偏差 (Look-ahead Bias)。

### 为什么需要 PIT 数据？

```
传统数据处理:
时间线: t1 ---- t2 ---- t3 ---- t4
数据版本:   [使用最新数据，包含未来信息] ❌

PIT 数据处理:
时间线: t1 ---- t2 ---- t3 ---- t4
数据版本: v1 ---- v2 ---- v3 ---- v4
         [每个时间点使用当时可获得的数据] ✓
```

### PIT 数据类型

| 类型 | 说明 | 示例 |
|------|------|------|
| 财务数据 | 财务报告发布时的数据 | 营收、净利润 |
| 行业分类 | 历史行业分类变更 | 申万行业分类 |
| 指数成分 | 历史成分股调整 | 沪深300成分 |
| 复权因子 | 历史复权因子变化 | 分红、拆股 |

In [ ]:
import qlib
from qlib.data import D
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 初始化 qlib
qlib.init(
    provider_uri="~/.qlib/qlib_data/cn_data",
    region="cn",
)

print("Qlib 初始化成功")

## 6.2 PIT 数据访问

### 6.2.1 检查 PIT 数据可用性

In [ ]:
from pathlib import Path

# 检查数据目录结构
data_dir = Path.home() / ".qlib" / "qlib_data" / "cn_data"

print("数据目录结构:")
for item in sorted(data_dir.iterdir()):
    if item.is_dir():
        print(f"  📁 {item.name}/")
        # 查看子目录内容
        sub_items = list(item.iterdir())[:3]
        for sub in sub_items:
            print(f"      - {sub.name}")
        if len(list(item.iterdir())) > 3:
            print(f"      ... ({len(list(item.iterdir())) - 3} more)")

In [ ]:
# 尝试获取 PIT 数据
# 注意：标准 cn_data 可能不包含完整 PIT 数据
# 这里演示如何检查和使用

try:
    # 尝试获取带 PIT 标记的数据
    # 使用 RefDate 操作符可以获取 PIT 数据
    df = D.features(
        instruments="SH600000",
        fields=["$close", "$volume"],
        start_time="2022-01-01",
        end_time="2022-12-31",
    )
    print("成功获取基础数据")
    print(f"数据形状: {df.shape}")
except Exception as e:
    print(f"获取数据时出错: {e}")

## 6.3 数据重采样

重采样是指将数据从一个时间频率转换到另一个时间频率，是量化分析中的常见操作。

### 6.3.1 重采样场景

```
高频 → 低频（降采样）:
- 分钟数据 → 日数据
- 日数据 → 周数据
- 日数据 → 月数据

低频 → 高频（升采样）:
- 月数据 → 日数据（需要填充）
- 季度财务数据 → 日数据
```

In [ ]:
# 获取日频数据
df_day = D.features(
    instruments="SH600000",
    fields=["$close", "$open", "$high", "$low", "$volume"],
    start_time="2022-01-01",
    end_time="2022-12-31",
)

print(f"日频数据形状: {df_day.shape}")
df_day.head()

In [ ]:
# 使用 qlib 的日历获取不同频率
calendar_day = D.calendar(freq="day", start_time="2022-01-01", end_time="2022-03-31")
calendar_week = D.calendar(freq="week", start_time="2022-01-01", end_time="2022-03-31")
calendar_month = D.calendar(freq="month", start_time="2022-01-01", end_time="2022-03-31")

print(f"日频交易日: {len(calendar_day)} 天")
print(f"周频: {len(calendar_week)} 周")
print(f"月频: {len(calendar_month)} 月")

print("\n日历示例:")
print(f"日频前5个: {calendar_day[:5]}")
print(f"周频: {calendar_week}")
print(f"月频: {calendar_month}")

In [ ]:
# 手动重采样：日频 → 周频
def resample_to_weekly(df):
    """将日频数据重采样为周频"""
    # 重置索引以便重采样
    df_reset = df.reset_index()
    df_reset['datetime'] = pd.to_datetime(df_reset['datetime'])
    df_reset.set_index('datetime', inplace=True)
    
    # 定义重采样规则
    weekly_df = df_reset.groupby('instrument').resample('W').agg({
        '$open': 'first',
        '$high': 'max',
        '$low': 'min',
        '$close': 'last',
        '$volume': 'sum',
    })
    
    return weekly_df

df_weekly = resample_to_weekly(df_day)
print(f"周频数据形状: {df_weekly.shape}")
df_weekly.head(10)

In [ ]:
# 手动重采样：日频 → 月频
def resample_to_monthly(df):
    """将日频数据重采样为月频"""
    df_reset = df.reset_index()
    df_reset['datetime'] = pd.to_datetime(df_reset['datetime'])
    df_reset.set_index('datetime', inplace=True)
    
    monthly_df = df_reset.groupby('instrument').resample('ME').agg({
        '$open': 'first',
        '$high': 'max',
        '$low': 'min',
        '$close': 'last',
        '$volume': 'sum',
    })
    
    return monthly_df

df_monthly = resample_to_monthly(df_day)
print(f"月频数据形状: {df_monthly.shape}")
df_monthly

In [ ]:
# 使用 qlib 内置的重采样工具
from qlib.utils.resam import resam_ts_data

# 查看重采样函数签名
import inspect
print("resam_ts_data 函数签名:")
print(inspect.signature(resam_ts_data))

## 6.4 滚动数据处理

滚动处理是量化分析中的核心操作，用于计算移动窗口统计量。

### 6.4.1 滚动操作原理

In [ ]:
# 演示滚动窗口计算
# 获取单只股票数据
df = D.features(
    instruments="SH600000",
    fields=["$close"],
    start_time="2022-01-01",
    end_time="2022-12-31",
)

# 计算各种滚动统计量
df['ma_5'] = df['$close'].rolling(5).mean()
df['ma_20'] = df['$close'].rolling(20).mean()
df['std_20'] = df['$close'].rolling(20).std()
df['max_20'] = df['$close'].rolling(20).max()
df['min_20'] = df['$close'].rolling(20).min()

print("滚动统计量计算结果:")
df.head(25)

In [ ]:
# 可视化滚动统计
plt.figure(figsize=(14, 8))

dates = df.index.get_level_values('datetime')

# 价格和均线
plt.subplot(2, 1, 1)
plt.plot(dates, df['$close'], label='收盘价', linewidth=1)
plt.plot(dates, df['ma_5'], label='MA5', linewidth=1, alpha=0.7)
plt.plot(dates, df['ma_20'], label='MA20', linewidth=1, alpha=0.7)
plt.fill_between(dates, df['ma_20'] - 2*df['std_20'], df['ma_20'] + 2*df['std_20'], 
                 alpha=0.2, label='±2σ 带宽')
plt.title('SH600000 - 收盘价与滚动统计')
plt.xlabel('日期')
plt.ylabel('价格')
plt.legend()
plt.grid(True, alpha=0.3)

# 波动率
plt.subplot(2, 1, 2)
plt.plot(dates, df['std_20'], label='20日波动率', color='orange')
plt.title('20日滚动波动率')
plt.xlabel('日期')
plt.ylabel('波动率')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6.4.2 滚动数据更新

In [ ]:
# 模拟滚动数据更新流程
class RollingDataUpdater:
    """滚动数据更新器"""
    
    def __init__(self, window_size=20):
        self.window_size = window_size
        self.data_buffer = []
    
    def update(self, new_value):
        """添加新数据点并计算滚动统计"""
        self.data_buffer.append(new_value)
        
        # 保持窗口大小
        if len(self.data_buffer) > self.window_size:
            self.data_buffer = self.data_buffer[-self.window_size:]
        
        # 计算统计量
        if len(self.data_buffer) >= self.window_size:
            return {
                'mean': np.mean(self.data_buffer),
                'std': np.std(self.data_buffer),
                'max': np.max(self.data_buffer),
                'min': np.min(self.data_buffer),
            }
        return None

# 测试滚动更新
updater = RollingDataUpdater(window_size=5)

print("滚动更新演示 (窗口=5):")
print("=" * 60)

test_values = [100, 102, 101, 103, 105, 104, 106, 108, 107, 109]
for i, val in enumerate(test_values):
    result = updater.update(val)
    print(f"第 {i+1:2d} 天: 值={val}, 滚动统计={result}")

## 6.5 扩展窗口与滚动窗口对比

```
滚动窗口 (Rolling Window):
窗口大小固定，随时间滑动
[1,2,3,4,5] → [2,3,4,5,6] → [3,4,5,6,7]

扩展窗口 (Expanding Window):
窗口大小递增，包含所有历史数据
[1] → [1,2] → [1,2,3] → [1,2,3,4]
```

In [ ]:
# 对比滚动窗口和扩展窗口
df['rolling_mean_20'] = df['$close'].rolling(20).mean()
df['expanding_mean'] = df['$close'].expanding().mean()

# 可视化对比
plt.figure(figsize=(14, 6))

dates = df.index.get_level_values('datetime')

plt.plot(dates, df['$close'], label='收盘价', linewidth=1, alpha=0.5)
plt.plot(dates, df['rolling_mean_20'], label='滚动均值 (20日)', linewidth=2)
plt.plot(dates, df['expanding_mean'], label='扩展均值', linewidth=2, linestyle='--')

plt.title('滚动窗口 vs 扩展窗口')
plt.xlabel('日期')
plt.ylabel('价格')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6.6 多股票滚动处理

In [ ]:
# 获取多只股票数据
df_multi = D.features(
    instruments="csi300",
    fields=["$close"],
    start_time="2022-10-01",
    end_time="2022-12-31",
)

print(f"多股票数据形状: {df_multi.shape}")
print(f"股票数量: {len(df_multi.index.get_level_values('instrument').unique())}")

In [ ]:
# 按股票分组计算滚动统计
def calculate_rolling_stats(df, window=20):
    """计算多股票滚动统计量"""
    # 按 instrument 分组
    grouped = df.groupby('instrument')
    
    # 计算滚动统计量
    result = df.copy()
    result['rolling_mean'] = grouped['$close'].transform(lambda x: x.rolling(window).mean())
    result['rolling_std'] = grouped['$close'].transform(lambda x: x.rolling(window).std())
    result['z_score'] = (result['$close'] - result['rolling_mean']) / result['rolling_std']
    
    return result

df_stats = calculate_rolling_stats(df_multi, window=20)

print("滚动统计计算完成")
df_stats.head(25)

In [ ]:
# 查看各股票的滚动统计结果
# 选取特定日期的数据
last_date = df_stats.index.get_level_values('datetime').max()
df_last = df_stats.xs(last_date, level='datetime')

print(f"\n{last_date} 各股票滚动统计:")
df_last.describe()

## 6.7 性能优化技巧

### 6.7.1 数据缓存

In [ ]:
import time

# 测试缓存效果
def test_cache_effect():
    """测试缓存对数据访问的影响"""
    
    # 第一次访问
    start = time.time()
    df1 = D.features(
        instruments="csi300",
        fields=["$close", "$volume"],
        start_time="2020-01-01",
        end_time="2022-12-31",
    )
    time1 = time.time() - start
    
    # 第二次访问（缓存）
    start = time.time()
    df2 = D.features(
        instruments="csi300",
        fields=["$close", "$volume"],
        start_time="2020-01-01",
        end_time="2022-12-31",
    )
    time2 = time.time() - start
    
    print(f"第一次访问: {time1:.4f} 秒")
    print(f"第二次访问: {time2:.4f} 秒 (缓存)")
    print(f"加速比: {time1/time2:.2f}x")
    
test_cache_effect()

### 6.7.2 并行处理

In [ ]:
# 演示并行处理
from multiprocessing import Pool
import time

def process_single_stock(stock_code):
    """处理单只股票数据"""
    try:
        df = D.features(
            instruments=stock_code,
            fields=["$close", "$volume"],
            start_time="2022-01-01",
            end_time="2022-12-31",
        )
        # 计算滚动统计
        df['ma_20'] = df['$close'].rolling(20).mean()
        return stock_code, len(df)
    except Exception as e:
        return stock_code, 0

# 获取股票列表
stocks = D.instruments(market="csi300")
stock_list = stocks['instrument'].tolist()[:20]  # 只取前20只演示

print(f"处理 {len(stock_list)} 只股票...")

# 串行处理
start = time.time()
results_serial = [process_single_stock(s) for s in stock_list]
time_serial = time.time() - start

print(f"串行处理时间: {time_serial:.2f} 秒")
print(f"处理结果: {dict(results_serial)}")

### 6.7.3 数据类型优化

In [ ]:
# 演示数据类型优化
df = D.features(
    instruments="csi300",
    fields=["$close", "$volume"],
    start_time="2022-01-01",
    end_time="2022-12-31",
)

# 原始内存使用
memory_original = df.memory_usage(deep=True).sum() / 1024**2  # MB

# 优化数据类型
df_optimized = df.copy()
for col in df_optimized.columns:
    if df_optimized[col].dtype == 'float64':
        df_optimized[col] = df_optimized[col].astype('float32')

# 优化后内存使用
memory_optimized = df_optimized.memory_usage(deep=True).sum() / 1024**2  # MB

print(f"原始内存使用: {memory_original:.2f} MB")
print(f"优化后内存使用: {memory_optimized:.2f} MB")
print(f"内存节省: {(1 - memory_optimized/memory_original)*100:.1f}%")

## 6.8 实践练习

### 练习目标

1. 加载 PIT 财务数据
2. 实现基于 PIT 数据的因子
3. 构建滚动数据更新流程
4. 性能对比分析

In [ ]:
# 练习1: 实现一个滚动波动率因子
# 计算过去 20 日的收益率波动率

# 你的代码



# 参考答案
# df = D.features(instruments='SH600000', fields=['$close'], start_time='2020-01-01', end_time='2022-12-31')
# df['return'] = df['$close'].pct_change()
# df['volatility_20d'] = df['return'].rolling(20).std() * np.sqrt(252)  # 年化波动率

In [ ]:
# 练习2: 实现周频数据获取和重采样
# 获取日频数据，然后重采样为周频
# 计算周收益率

# 你的代码



# 参考答案
# df_day = D.features(instruments='SH600000', fields=['$close'], start_time='2022-01-01', end_time='2022-12-31')
# df_weekly = resample_to_weekly(df_day)
# df_weekly['weekly_return'] = df_weekly['$close'].pct_change()

In [ ]:
# 练习3: 实现一个高效的多股票数据处理函数
# 批量获取沪深300成分股数据
# 计算每只股票的 20 日均线偏离度

# 你的代码



# 参考答案
# def calculate_ma_deviation(df, window=20):
#     grouped = df.groupby('instrument')
#     df['ma'] = grouped['$close'].transform(lambda x: x.rolling(window).mean())
#     df['ma_deviation'] = (df['$close'] - df['ma']) / df['ma']
#     return df

In [ ]:
# 练习4: 性能对比
# 对比串行和并行处理多股票数据的性能差异

# 你的代码



# 提示：使用 multiprocessing.Pool 进行并行处理

## 6.9 本章小结

本章我们学习了：

1. **PIT 数据**：
   - Point-in-Time 数据概念
   - 避免前视偏差

2. **数据重采样**：
   - 高频 → 低频（降采样）
   - 低频 → 高频（升采样）
   - 使用 qlib 日历和重采样工具

3. **滚动数据处理**：
   - 滚动窗口 vs 扩展窗口
   - 滚动统计量计算
   - 多股票滚动处理

4. **性能优化**：
   - 数据缓存
   - 并行处理
   - 数据类型优化

### 关键 API 速查

```python
# 获取不同频率日历
calendar_day = D.calendar(freq="day")
calendar_week = D.calendar(freq="week")

# 滚动计算
df['rolling_mean'] = df['$close'].rolling(20).mean()
df['rolling_std'] = df['$close'].rolling(20).std()

# 扩展计算
df['expanding_mean'] = df['$close'].expanding().mean()

# 分组滚动
grouped = df.groupby('instrument')
df['ma'] = grouped['$close'].transform(lambda x: x.rolling(20).mean())
```

### 下一部分预告

下一部分我们将学习模型训练与预测，包括：
- 模型体系架构
- 传统机器学习模型（LightGBM、XGBoost）
- 深度学习模型（LSTM、Transformer）
- 集成学习与模型融合